# 23 · Final fine-tuning — RT-DETRv2-L
Automatically loads notebook 13's best configuration and runs baseline/tuned seeds 17, 42, and 3407.

In [ ]:
DATASET_TRACK = "2class"
START_FINETUNING = False

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_PATH = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_PATH / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", "main", "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git", str(REPO_PATH)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_PATH), "--no-deps"], check=True)
else:
    REPO_PATH = Path.cwd()
sys.path.insert(0, str(REPO_PATH))
DRIVE_ROOT = Path(os.environ.get("VISDRONE_DRIVE_ROOT", "/content/drive/MyDrive/visdrone_architecture_benchmark"))

In [ ]:
MODEL_ID = "rtdetrv2_l"
from src.models.rtdetrv2.trainer import RTDetrSharedTrainer
print(f"Shared training engine: {RTDetrSharedTrainer.__name__}")
print("RT-DETR optimizer policy: rtdetr_recipe_v2")
if SMOKE_TEST:
    result = {"status": "guarded", "model_id": MODEL_ID}
else:
    from src.workflows.environment import ensure_model_environment
    from src.hpo.final_workflow import FinalExperimentWorkflow
    environment = (
        ensure_model_environment(MODEL_ID, REPO_PATH, DRIVE_ROOT)
        if START_FINETUNING
        else {"status": "SKIPPED_PREVIEW", "family": "rtdetr"}
    )
    result = FinalExperimentWorkflow(REPO_PATH, DRIVE_ROOT, MODEL_ID, DATASET_TRACK).run(start_expensive_stage=START_FINETUNING)
result